# 03 · Statistical forecasts (Theta + AutoETS + SeasonalNaive)

Drives `m5.cv.stats_cv` — the same path `make cv-stats` uses. 
Outputs the rolling-origin CV frame to `artifacts/cv_stats.parquet` so 
`06_score.ipynb` picks it up automatically.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import stats_cv
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.models.stats import fit_predict_stats
from m5.plots import configure_style

configure_style()
set_global_seed()

42

## Cross-validation

In [3]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')

cv = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
cv.to_parquet(SETTINGS.artifacts_dir / 'cv_stats.parquet', index=False)
cv.tail()

12:51:02 | INFO    | m5.cv:stats_cv:46 - stats_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/Git

,unique_id,ds,cutoff,y,Theta,AutoETS,SeasonalNaive
419995,HOUSEHOLD_2_516_WI_3,2016-05-18,2016-04-24,0.0,-0.010341,0.080732,0.0
419996,HOUSEHOLD_2_516_WI_3,2016-05-19,2016-04-24,0.0,-0.010730,0.078874,0.0
419997,HOUSEHOLD_2_516_WI_3,2016-05-20,2016-04-24,0.0,-0.011120,0.080220,0.0
419998,HOUSEHOLD_2_516_WI_3,2016-05-21,2016-04-24,0.0,-0.011509,0.160451,0.0
419999,HOUSEHOLD_2_516_WI_3,2016-05-22,2016-04-24,0.0,-0.011898,0.121099,0.0


In [4]:
components = compute_components(df[df['ds'] < cv['ds'].min()])
wrmsse_for_models(cv[['unique_id', 'ds', 'y']], cv, components)

AutoETS          0.825115
Theta            0.833672
SeasonalNaive    1.048186
Name: wrmsse, dtype: float64

## Train on full data, emit a 28-day forecast

One call per Nixtla — same path as `make forecast-stats`.

In [5]:
forecast = fit_predict_stats(df, horizon=SETTINGS.horizon)
forecast.to_parquet(SETTINGS.forecasts_dir / 'forecast_stats.parquet', index=False)
forecast.head()

12:51:14 | INFO    | m5.models.stats:fit_predict_stats:59 - fit_predict_stats: Theta+AutoETS+SeasonalNaive on 5,000 series, h=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide


,unique_id,ds,Theta,AutoETS,SeasonalNaive
0,FOODS_1_001_CA_1,2016-05-23,0.626201,0.903332,0.0
1,FOODS_1_001_CA_1,2016-05-24,0.626839,0.969149,0.0
2,FOODS_1_001_CA_1,2016-05-25,0.627478,0.826899,0.0
3,FOODS_1_001_CA_1,2016-05-26,0.628116,0.647394,1.0
4,FOODS_1_001_CA_1,2016-05-27,0.628754,0.882315,0.0


## Optional — Kaggle-format submission per model

Pivot a single model column into the wide submission layout.

In [6]:
model = 'AutoETS'
submission = make_submission(
    forecast[['unique_id', 'ds', model]], h=SETTINGS.horizon, value_col=model, layout='kaggle',
)
submission.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
unique_id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0.903332,0.969149,0.826899,0.647394,0.882315,0.801520,0.620614,0.903332,0.969149,0.826899,...,0.882315,0.801520,0.620614,0.903332,0.969149,0.826899,0.647394,0.882315,0.801520,0.620614
FOODS_1_001_TX_2,0.213145,0.534344,0.657622,0.518012,0.380541,0.654626,0.621280,0.213145,0.534344,0.657622,...,0.380541,0.654626,0.621280,0.213145,0.534344,0.657622,0.518012,0.380541,0.654626,0.621280
FOODS_1_003_CA_4,0.251302,0.291953,0.240179,0.343820,0.447224,0.208070,0.067668,0.251302,0.291953,0.240179,...,0.447224,0.208070,0.067668,0.251302,0.291953,0.240179,0.343820,0.447224,0.208070,0.067668
FOODS_1_003_TX_3,0.006109,0.149686,0.175738,0.078988,0.039094,0.039031,0.039256,0.006109,0.149686,0.175738,...,0.039094,0.039031,0.039256,0.006109,0.149686,0.175738,0.078988,0.039094,0.039031,0.039256
FOODS_1_003_WI_1,0.642046,1.498692,0.930863,1.660267,1.377832,1.654184,0.861688,0.642046,1.498692,0.930863,...,1.377832,1.654184,0.861688,0.642046,1.498692,0.930863,1.660267,1.377832,1.654184,0.861688
